# Emberi beavatkozással működő munkafolyamat a Microsoft Agent Framework segítségével

## 🎯 Tanulási célok

Ebben a jegyzetfüzetben megtanulod, hogyan valósíthatók meg **emberi beavatkozással működő** munkafolyamatok a Microsoft Agent Framework `RequestInfoExecutor` segítségével. Ez az erőteljes minta lehetővé teszi, hogy az MI munkafolyamatok megálljanak az emberi input begyűjtése érdekében, így interaktívvá téve az ügynököket, és lehetőséget adva az embereknek a kritikus döntések irányítására.

## 🔄 Mi az a Human-in-the-Loop?

A **human-in-the-loop (HITL)** egy olyan tervezési minta, ahol az MI ügynökök megállnak a végrehajtásban, hogy emberektől kérjenek bevitelt, mielőtt folytatnák. Ez fontos az alábbi esetekben:

- ✅ **Kritikus döntések** - Emberi jóváhagyás beszerzése fontos lépések előtt
- ✅ **Kétértelmű helyzetek** - Az emberek segítenek tisztázni, ha az MI bizonytalan
- ✅ **Felhasználói preferenciák** - Kérdezd meg a felhasználót több lehetőség közötti választásról
- ✅ **Megfelelőség és biztonság** - Biztosíts emberi felügyeletet szabályozott műveleteknél
- ✅ **Interaktív élmények** - Valósíts meg beszélgető ügynököket, amelyek a felhasználói inputokra reagálnak

## 🏗️ Hogyan működik a Microsoft Agent Framework-ben

A keretrendszer három kulcsfontosságú elemet kínál a HITL-hez:

1. **`RequestInfoExecutor`** - Egy speciális végrehajtó, amely megállítja a munkafolyamatot és kibocsát egy `RequestInfoEvent` eseményt
2. **`RequestInfoMessage`** - Alaposztály az embernek küldött típusos kérelmi adatokhoz
3. **`RequestResponse`** - Összekapcsolja az emberi válaszokat az eredeti kérésekkel `request_id` alapján

**Munkafolyamat minta:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Példánk: Szállásfoglalás felhasználói megerősítéssel

A feltételes munkafolyamatra építve hozzáadunk egy emberi megerősítést **előtte**, mielőtt alternatív úti célokat javasolnánk:

1. A felhasználó kér egy úti célt (pl. "Párizs")
2. Az `availability_agent` ellenőrzi a szabad szobákat
3. **Ha nincs szabad szoba** → a `confirmation_agent` megkérdezi: "Szeretne alternatívákat látni?"
4. A munkafolyamat **megáll** a `RequestInfoExecutor` segítségével
5. **Az ember válaszol** "igen" vagy "nem" a konzolon
6. A `decision_manager` a válasz alapján irányít:
   - **Igen** → Mutassa az alternatív úti célokat
   - **Nem** → Törölje a foglalási kérelmet
7. Mutassa a végső eredményt

Ez megmutatja, hogyan adhatsz a felhasználóknak kontrollt az ügynök javaslatai felett!

---

Kezdjünk neki! 🚀


## 1. lépés: Szükséges könyvtárak importálása

Importáljuk az alap Agent Framework komponenseket, plusz a **human-in-the-loop specifikus osztályokat**:
- `RequestInfoExecutor` - Végrehajtó, amely szünetelteti a munkafolyamatot emberi bevitelért
- `RequestInfoEvent` - Esemény, amely emberi bevitel kérésekor jön létre
- `RequestInfoMessage` - Alaposztály típusosított kérés terhelésekhez
- `RequestResponse` - Összekapcsolja az emberi válaszokat a kérésekkel
- `WorkflowOutputEvent` - Esemény a munkafolyamat kimeneteinek detektálására


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## 2. lépés: Pydantic modellek definiálása strukturált kimenetekhez

Ezek a modellek határozzák meg a **sémát**, amit az ügynökök visszaadnak. Megtartjuk az összes modellt a feltételes munkafolyamatból, és hozzáadjuk:

**Új az emberi bevitelhez:**
- `HumanFeedbackRequest` - A `RequestInfoMessage` alosztálya, amely meghatározza az embereknek küldött kérés csomagot
  - Tartalmazza a `prompt` (feltenendő kérdés) és a `destination` (információ a nem elérhető városról) mezőket


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## 3. lépés: Hozza létre a szállásfoglaló eszközt

Ugyanaz az eszköz, mint a feltételes munkafolyamatban - ellenőrzi, hogy vannak-e szabad szobák az adott úticélon.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## 4. lépés: Feltételes függvények meghatározása az irányításhoz

Négy **feltételes függvényre** van szükségünk az ember a folyamatban munkafolyamatunkhoz:

**A feltételes munkafolyamatból:**
1. `has_availability_condition` - Irányít, amikor szállodák ELÉRHETŐK
2. `no_availability_condition` - Irányít, amikor szállodák NEM ELÉRHETŐK

**Új az ember a folyamatban esetén:**
3. `user_wants_alternatives_condition` - Irányít, amikor a felhasználó "igen"-t mond az alternatívákra
4. `user_declines_alternatives_condition` - Irányít, amikor a felhasználó "nem"-et mond az alternatívákra


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## 5. lépés: Hozd létre a Decision Manager végrehajtót

Ez az **emberi beavatkozást igénylő minta magja**! A `DecisionManager` egy egyedi `Executor`, amely:

1. **Fogadja az emberi visszajelzést** `RequestResponse` objektumokon keresztül
2. **Feldolgozza a felhasználó döntését** (igen/nem)
3. **Irányítja a munkafolyamatot** azzal, hogy üzeneteket küld a megfelelő ügynököknek

Fő jellemzők:
- `@handler` dekorátort használ a metódusok munkafolyamat-lépésként való megjelenítéséhez
- Fogad `RequestResponse[HumanFeedbackRequest, str]` objektumokat, amelyek tartalmazzák az eredeti kérést és a felhasználó válaszát
- Egyszerű "igen" vagy "nem" üzeneteket ad, amelyek meghívják feltételfüggvényeinket


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## 6. lépés: Egyéni megjelenítési végrehajtó létrehozása

Ugyanaz a megjelenítési végrehajtó, mint a feltételes munkafolyamatnál – a végső eredményeket munkafolyamat kimeneteként adja vissza.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## 7. lépés: Környezeti változók betöltése

Konfigurálja az LLM klienst (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## 8. lépés: AI ügynökök és végrehajtók létrehozása

Létrehozunk **hat munkafolyamat-komponenst**:

**Ügynökök (AgentExecutor-be csomagolva):**
1. **availability_agent** - Ellenőrzi a szálloda elérhetőségét az eszköz segítségével
2. **confirmation_agent** - 🆕 Előkészíti az emberi megerősítési kérést
3. **alternative_agent** - Alternatív városokat javasol (amikor a felhasználó igent mond)
4. **booking_agent** - Foglalásra ösztönöz (amikor vannak szabad szobák)
5. **cancellation_agent** - 🆕 Kezeli a lemondási üzenetet (amikor a felhasználó nemet mond)

**Speciális végrehajtók:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, amely szünetelteti a munkafolyamatot emberi bevitelért
7. **decision_manager** - 🆕 Egyedi végrehajtó, amely az emberi válasz alapján irányít (már fent definiálva)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## 9. lépés: A munkafolyamat elkészítése emberi beavatkozással

Most felépítjük a munkafolyamat gráfját **feltételes irányítással**, amely tartalmazza az emberi beavatkozás útvonalát:

**Munkafolyamat felépítése:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Fontos élek:**
- `availability_agent → confirmation_agent` (amikor nincs szoba)
- `confirmation_agent → prepare_human_request` (típus átalakítása)
- `prepare_human_request → request_info_executor` (szünet az emberi beavatkozásra)
- `request_info_executor → decision_manager` (mindig - RequestResponse-t ad)
- `decision_manager → alternative_agent` (amikor a felhasználó "igen"-t mond)
- `decision_manager → cancellation_agent` (amikor a felhasználó "nem"-et mond)
- `availability_agent → booking_agent` (amikor szobák elérhetők)
- Minden útvonal a `display_result`-ben ér véget


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## 10. lépés: Futtassa az 1. tesztesetet – Város ELÉRHETŐSÉG NÉLKÜL (Párizs emberi megerősítéssel)

Ez a teszt bemutatja a **teljes emberi visszacsatolásos ciklust**:

1. Szállás kérése Párizsban
2. availability_agent ellenőrzi → Nincs szoba
3. confirmation_agent létrehozza az emberi kérdést
4. request_info_executor **szünetelteti a munkafolyamatot** és kibocsátja a `RequestInfoEvent`-et
5. **Az alkalmazás érzékeli az eseményt és felkéri a felhasználót a konzolon**
6. A felhasználó beírja, hogy „igen” vagy „nem”
7. Az alkalmazás elküldi a választ a `send_responses_streaming()` segítségével
8. decision_manager a válasz alapján irányít
9. A végső eredmény megjelenik

**Fő minta:**
- Használja a `workflow.run_stream()` parancsot az első iterációnál
- Használja a `workflow.send_responses_streaming(pending_responses)` parancsot a további iterációknál
- Figyelje a `RequestInfoEvent` eseményt, hogy érzékelje, mikor van szükség emberi bevitelre
- Figyelje a `WorkflowOutputEvent` eseményt a végső eredmények rögzítéséhez


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## 11. lépés: Futassuk a 2. tesztesetet – Város ELÉRHETŐSÉGGEL (Stockholm – Nincs szükség emberi beavatkozásra)

Ez a teszt bemutatja a **közvetlen utat**, amikor szobák elérhetők:

1. Kérés egy szállodára Stockholmban
2. availability_agent ellenőrzi → Szobák elérhetők ✅
3. booking_agent foglalást javasol
4. display_result megjeleníti a visszaigazolást
5. **Nincs szükség emberi beavatkozásra!**

Az munkafolyamat teljesen megkerüli az emberi beavatkozás útját, ha szobák elérhetők.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Főbb tanulságok és a "human-in-the-loop" legjobb gyakorlatai

### ✅ Amit megtanultál:

#### 1. **RequestInfoExecutor minta**
A Microsoft Agent Framework "human-in-the-loop" mintája három kulcsfontosságú elemet használ:
- `RequestInfoExecutor` - Megállítja a munkafolyamatot és eseményeket bocsát ki
- `RequestInfoMessage` - Tipizált kérés törzs alap osztálya (ezt kell leszármaztatni!)
- `RequestResponse` - Összekapcsolja az emberi válaszokat az eredeti kérésekkel

**Fontos megértés:**
- A `RequestInfoExecutor` NEM gyűjt be adatot maga - csak megállítja a munkafolyamatot
- Az alkalmazásodnak kell hallgatnia a `RequestInfoEvent` eseményeket és begyűjtenie a bevitelt
- Hívnod kell a `send_responses_streaming()` metódust egy szótárral, amely a `request_id`-t a felhasználó válaszához társítja

#### 2. **Streaming végrehajtási minta**
```python
# Első iteráció
stream = workflow.run_stream(initial_request)

# Utána következő iterációk (emberi bevitelt követően)
stream = workflow.send_responses_streaming(pending_responses)

# Mindig események feldolgozása
events = [event async for event in stream]
```

#### 3. **Eseményvezérelt architektúra**
Hallgass meg adott eseményeket a munkafolyamat irányításához:
- `RequestInfoEvent` - Emberi bevitel szükséges (munkafolyamat szünetel)
- `WorkflowOutputEvent` - Végső eredmény elérhető (munkafolyamat befejeződött)
- `WorkflowStatusEvent` - Állapotváltozások (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS, stb.)

#### 4. **Egyedi végrehajtók @handler-rel**
A `DecisionManager` bemutatja, hogyan lehet létrehozni végrehajtókat, amelyek:
- Az `@handler` dekorátort használják, hogy a metódusokat munkafolyamat lépésekként tegye elérhetővé
- Tipizált üzeneteket fogadnak (pl. `RequestResponse[HumanFeedbackRequest, str]`)
- Más végrehajtókhoz üzenetek küldésével irányítják a munkafolyamatot
- Hozzáférnek a kontextushoz a `WorkflowContext` segítségével

#### 5. **Feltételes átirányítás emberi döntések alapján**
Létrehozhatsz olyan feltétel funkciókat, amelyek értékelik az emberi válaszokat:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Valós alkalmazások:

1. **Jóváhagyási munkafolyamatok**
   - Számviteli költségelszámolások előtti vezetői jóváhagyás
   - Emberi felülvizsgálat szükséges automatizált e-mailek küldése előtt
   - Nagy összegű tranzakciók megerősítése végrehajtás előtt

2. **Tartalom moderálás**
   - Kétes tartalom megjelölése emberi felülvizsgálatra
   - Moderátoroknak hagyni a végső döntést kétes esetekben
   - Emberi beavatkozás, ha az AI bizonyossága alacsony

3. **Ügyfélszolgálat**
   - Az AI kezelje automatikusan a rutin kérdéseket
   - Bonyolult problémákat továbbíts emberi ügynököknek
   - Kérdezd meg az ügyfelet, hogy akar-e emberrel beszélni

4. **Adatfeldolgozás**
   - Emberi közbelépés kétértelmű adatok esetén
   - AI értelmezések megerősítése nem világos dokumentumoknál
   - Felhasználók választhassanak több érvényes értelmezés között

5. **Biztonságkritikus rendszerek**
   - Emberi megerősítés kötelező visszafordíthatatlan műveletek előtt
   - Jóváhagyás érzékeny adatok eléréséhez
   - Döntések megerősítése szabályozott iparágakban (egészségügy, pénzügy)

6. **Interaktív ügynökök**
   - Beszélgető botok készítése, amelyek további kérdéseket tesznek fel
   - Varázslók létrehozása, amelyek végigvezetik a felhasználókat összetett folyamatokon
   - Ügynökök tervezése, amelyek lépésről lépésre együttműködnek az emberekkel

### 🔄 Összehasonlítás: Emberi beavatkozással vs. anélkül

| Tulajdonság | Feltételes munkafolyamat | Human-in-the-Loop munkafolyamat |
|---------|---------------------|---------------------------|
| **Futtatás** | Egyszeri `workflow.run()` | Ciklus `run_stream()` + `send_responses_streaming()` |
| **Felhasználói beviteli mód** | Nincs (teljesen automatizált) | Interaktív promptok `input()` vagy UI segítségével |
| **Komponensek** | Ügynökök + végrehajtók | + RequestInfoExecutor + DecisionManager |
| **Események** | Csak AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent, stb. |
| **Szüneteltetés** | Nincs szüneteltetés | Munkafolyamat megáll RequestInfoExecutornál |
| **Emberi irányítás** | Nincs emberi irányítás | Az emberek hozzák meg a kulcsdöntéseket |
| **Alkalmazási terület** | Automatizálás | Együttműködés és felügyelet |

### 🚀 Fejlett minták:

#### Többszörös emberi döntési pontok
Több `RequestInfoExecutor` csomópont is lehet ugyanabban a munkafolyamatban:
```python
.add_edge(agent1, request_info_1)  # Első emberi döntés
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Második emberi döntés
.add_edge(decision_manager_2, final_agent)
```

#### Időtúllépés kezelése
Időtúllépések implementálása emberi válaszok esetére:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Alapértelmezettként a biztonságos lehetőséget választja
```

#### Gazdag UI integráció
Az `input()` helyett integrálj webes UI-t, Slack-et, Teams-et stb.:
```python
if isinstance(event, RequestInfoEvent):
    # Értesítés küldése a felhasználó által preferált csatornára
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Feltételes human-in-the-loop
Csak bizonyos helyzetekben kérj emberi bevitelt:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Csak akkor irányítsa emberhez, ha az önbizalom alacsony vagy az érték magas
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Legjobb gyakorlatok:

1. **Mindig leszármaztasd a RequestInfoMessage osztályt**
   - Tipusbiztonságot és validálást biztosít
   - Gazdag kontextust nyújt UI megjelenítéshez
   - Tisztázza az egyes kérés típusok szándékát

2. **Használj kifejező promptokat**
   - Adj meg kontextust arról, hogy mit kérdezel
   - Magyarázd el a választási lehetőségek következményeit
   - Tartsd egyszerűnek és világosnak a kérdéseket

3. **Kezeld a váratlan beviteleket**
   - Érvényesítsd a felhasználói válaszokat
   - Adj alapértelmezett értékeket érvénytelen bevitel esetén
   - Adj egyértelmű hibajelzéseket

4. **Kövesd nyomon a kérés azonosítókat**
   - Használd a request_id és válaszok közötti összefüggést
   - Ne próbáld manuálisan kezelni az állapotot

5. **Ne blokkolj**
   - Ne blokkolj szálakat bevitel várásakor
   - Használj aszinkron mintákat végig
   - Támogasd a párhuzamos munkafolyamat példányokat

### 📚 Kapcsolódó fogalmak:

- **Agent Middleware** - Ügynök hívások elfogása (előző jegyzetfüzet)
- **Munkafolyamat állapot kezelése** - Munkafolyamat állapotának perzisztálása futtatások között
- **Több ügynök együttműködés** - Emberi beavatkozás kombinálása ügynök csapatokkal
- **Eseményvezérelt architektúrák** - Reaktív rendszerek építése események alapján

---

### 🎓 Gratulálunk!

Mesterien elsajátítottad a human-in-the-loop munkafolyamatokat a Microsoft Agent Framework-ben! Most már tudod, hogyan kell:
- ✅ Megállítani a munkafolyamatot emberi bevitel gyűjtéséhez
- ✅ Használni a RequestInfoExecutort és RequestInfoMessage-t
- ✅ Kezelni streaming végrehajtást eseményekkel
- ✅ Egyedi végrehajtókat létrehozni @handler-rel
- ✅ Munkafolyamatokat emberi döntések alapján irányítani
- ✅ Interaktív AI ügynököket építeni, amelyek együttműködnek emberekkel

**Ez egy kritikus minta a megbízható, irányítható AI rendszerek építéséhez!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
